企业级最佳实践
================

```mermaid
flowchart TB
    subgraph 安全层
        S1[输入过滤] --> S2[数据加密]
        S2 --> S3[访问控制]
    end
    subgraph 隐私层
        P1[PII识别脱敏] --> P2[数据生命周期管理]
    end
    subgraph 性能层
        PF1[缓存策略] --> PF2[异步处理]
        PF2 --> PF3[负载均衡]
    end
    subgraph 监控层
        M1[日志系统] --> M2[指标监控 Prometheus]
        M2 --> M3[健康检查]
    end
    subgraph 可用层
        HA1[故障恢复] --> HA2[熔断降级]
    end
    安全层 --> 隐私层
    隐私层 --> 性能层
    性能层 --> 监控层
    监控层 --> 可用层



In [ ]:

## 一、安全合规

### 1.1 输入输出过滤

恶意输入可能导致Prompt注入攻击，需要建立严格的输入输出过滤机制。




import re
from typing import Optional

def sanitize_input(input_text: str) -> str:
    patterns = [
        r'(?i)system\s*prompt',
        r'(?i)ignore.*previous',
        r'(?i)reset.*instructions',
        r'(?i)override.*settings',
    ]
    
    for pattern in patterns:
        input_text = re.sub(pattern, '[REDACTED]', input_text)
    
    return input_text[:4096]

def validate_output(output_text: str) -> Optional[str]:
    forbidden_patterns = [
        r'(?i)execute.*command',
        r'(?i)rm\s*-rf',
        r'(?i)curl.*http',
    ]
    
    for pattern in forbidden_patterns:
        if re.search(pattern, output_text):
            return None
    
    return output_text



In [ ]:

### 1.2 数据加密与传输安全




from cryptography.fernet import Fernet
from typing import Dict

class SecureDataManager:
    def __init__(self, key: bytes = None):
        self.key = key or Fernet.generate_key()
        self.cipher = Fernet(self.key)
    
    def encrypt_data(self, data: Dict) -> bytes:
        import json
        data_str = json.dumps(data)
        return self.cipher.encrypt(data_str.encode())
    
    def decrypt_data(self, encrypted_data: bytes) -> Dict:
        import json
        data_str = self.cipher.decrypt(encrypted_data).decode()
        return json.loads(data_str)



In [ ]:

### 1.3 访问控制与权限管理




from enum import Enum
from typing import Set

class Role(Enum):
    ADMIN = "admin"
    USER = "user"
    GUEST = "guest"

class PermissionManager:
    def __init__(self):
        self.permissions: Dict[Role, Set[str]] = {
            Role.ADMIN: {"read", "write", "delete", "configure"},
            Role.USER: {"read", "write"},
            Role.GUEST: {"read"}
        }
    
    def has_permission(self, role: Role, action: str) -> bool:
        return action in self.permissions.get(role, set())



In [ ]:

## 二、数据隐私保护

### 2.1 PII数据识别与脱敏




import spacy
from typing import List

class PIIProcessor:
    def __init__(self):
        self.nlp = spacy.load("zh_core_web_sm")
    
    def identify_pii(self, text: str) -> List[Dict]:
        doc = self.nlp(text)
        pii_entities = []
        
        for ent in doc.ents:
            if ent.label_ in ["PERSON", "ORG", "GPE", "PHONE", "EMAIL"]:
                pii_entities.append({
                    "text": ent.text,
                    "label": ent.label_,
                    "start": ent.start_char,
                    "end": ent.end_char
                })
        
        return pii_entities
    
    def anonymize_text(self, text: str) -> str:
        pii_entities = self.identify_pii(text)
        result = text
        
        for entity in sorted(pii_entities, key=lambda x: x["start"], reverse=True):
            replacement = f"[{entity['label']}]"
            result = result[:entity["start"]] + replacement + result[entity["end"]:]
        
        return result



In [ ]:

### 2.2 数据生命周期管理




from datetime import datetime, timedelta
from typing import Dict, Any

class DataLifecycleManager:
    def __init__(self, retention_days: int = 90):
        self.retention_days = retention_days
    
    def is_expired(self, created_at: datetime) -> bool:
        return datetime.now() - created_at > timedelta(days=self.retention_days)
    
    def purge_expired_data(self, records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        return [r for r in records if not self.is_expired(r.get("created_at"))]



In [ ]:

### 2.3 GDPR与合规检查清单

- [ ] 数据最小化原则
- [ ] 用户同意机制
- [ ] 数据主体权利（访问、更正、删除）
- [ ] 数据保护影响评估（DPIA）
- [ ] 第三方处理器合同
- [ ] 跨境数据传输合规

## 三、成本优化策略

### 3.1 API调用成本监控




from dataclasses import dataclass
from typing import Dict, List
from datetime import datetime

@dataclass
class APICallRecord:
    endpoint: str
    model: str
    tokens_used: int
    cost_usd: float
    timestamp: datetime

class CostMonitor:
    def __init__(self):
        self.records: List[APICallRecord] = []
        self.daily_budget: float = 100.0
    
    def add_record(self, record: APICallRecord):
        self.records.append(record)
    
    def get_daily_cost(self, date: datetime = None) -> float:
        target_date = date or datetime.now()
        day_start = target_date.replace(hour=0, minute=0, second=0)
        day_end = day_start + timedelta(days=1)
        
        return sum(
            r.cost_usd for r in self.records
            if day_start <= r.timestamp < day_end
        )
    
    def is_over_budget(self) -> bool:
        return self.get_daily_cost() >= self.daily_budget



In [ ]:

### 3.2 模型选择策略




from enum import Enum

class ModelTier(Enum):
    ECONOMY = "economy"
    STANDARD = "standard"
    PREMIUM = "premium"

class ModelSelector:
    def __init__(self):
        self.tier_config = {
            ModelTier.ECONOMY: {"model": "gpt-3.5-turbo", "cost_per_k": 0.0015},
            ModelTier.STANDARD: {"model": "gpt-4", "cost_per_k": 0.03},
            ModelTier.PREMIUM: {"model": "gpt-4-turbo", "cost_per_k": 0.01}
        }
    
    def select_model(self, task_complexity: str) -> str:
        if task_complexity == "simple":
            return self.tier_config[ModelTier.ECONOMY]["model"]
        elif task_complexity == "medium":
            return self.tier_config[ModelTier.STANDARD]["model"]
        else:
            return self.tier_config[ModelTier.PREMIUM]["model"]



In [ ]:

### 3.3 缓存策略实现




from functools import lru_cache
from typing import Any, Callable

def cached_llm_response(maxsize: int = 1024):
    def decorator(func: Callable) -> Callable:
        @lru_cache(maxsize=maxsize)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator

@cached_llm_response(maxsize=512)
def get_llm_response(prompt: str, model: str) -> str:
    pass



In [ ]:

## 四、性能调优

### 4.1 请求批处理




from typing import List, Dict, Any
import asyncio

async def batch_process_requests(
    requests: List[Dict[str, Any]],
    batch_size: int = 10
) -> List[Any]:
    results = []
    
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        tasks = [process_request(req) for req in batch]
        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
    
    return results



In [ ]:

### 4.2 异步处理模式




import asyncio
from typing import Coroutine, List

class AsyncTaskManager:
    def __init__(self, max_concurrent: int = 5):
        self.semaphore = asyncio.Semaphore(max_concurrent)
    
    async def execute_with_limit(self, coro: Coroutine) -> Any:
        async with self.semaphore:
            return await coro
    
    async def execute_all(self, coros: List[Coroutine]) -> List[Any]:
        tasks = [self.execute_with_limit(c) for c in coros]
        return await asyncio.gather(*tasks)



In [ ]:

### 4.3 负载均衡配置




from random import random
from typing import List

class LoadBalancer:
    def __init__(self, endpoints: List[str]):
        self.endpoints = endpoints
        self.weights = [1.0 / len(endpoints)] * len(endpoints)
    
    def select_endpoint(self) -> str:
        r = random()
        cumulative = 0.0
        
        for i, weight in enumerate(self.weights):
            cumulative += weight
            if r < cumulative:
                return self.endpoints[i]
        
        return self.endpoints[-1]



In [ ]:

## 五、监控与可观测性

### 5.1 日志系统设计




import logging
from logging.handlers import RotatingFileHandler

def setup_logging():
    logger = logging.getLogger("agent_system")
    logger.setLevel(logging.INFO)
    
    handler = RotatingFileHandler(
        "agent.log",
        maxBytes=10*1024*1024,
        backupCount=5
    )
    
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    
    return logger



In [ ]:

### 5.2 指标监控




from prometheus_client import Counter, Histogram, Gauge

class MetricsCollector:
    def __init__(self):
        self.requests_total = Counter(
            "agent_requests_total",
            "Total number of requests",
            ["endpoint", "status"]
        )
        
        self.request_duration = Histogram(
            "agent_request_duration_seconds",
            "Request duration in seconds"
        )
        
        self.active_tasks = Gauge(
            "agent_active_tasks",
            "Number of active tasks"
        )
    
    def record_request(self, endpoint: str, status: str, duration: float):
        self.requests_total.labels(endpoint=endpoint, status=status).inc()
        self.request_duration.observe(duration)



graph TB
    LB[负载均衡器] --> S1[Agent服务实例 1]
    LB --> S2[Agent服务实例 2]
    LB --> S3[Agent服务实例 3]
    S1 --> Cache[Redis缓存]
    S2 --> Cache
    S3 --> Cache
    S1 --> DB[(数据库)]
    S2 --> DB
    S3 --> DB
    S1 --> LLM[LLM API]
    S2 --> LLM
    S3 --> LLM



In [ ]:

## 六、高可用性架构

### 6.1 故障恢复机制




from tenacity import retry, stop_after_attempt, wait_exponential

class ResilientClient:
    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10)
    )
    async def make_request(self, url: str) -> Any:
        response = await self.http_client.get(url)
        response.raise_for_status()
        return response.json()



In [ ]:

### 6.2 健康检查端点




from fastapi import FastAPI, Response, status

app = FastAPI()

@app.get("/health")
async def health_check():
    checks = {
        "database": check_database(),
        "api_service": check_api_service(),
        "cache": check_cache()
    }
    
    if all(checks.values()):
        return {"status": "healthy", "checks": checks}
    else:
        return Response(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            content={"status": "unhealthy", "checks": checks}
        )



In [ ]:

## 实践练习

1. 实现一个完整的安全输入过滤系统
2. 设计一个成本监控仪表盘
3. 搭建一个高可用性的Agent服务架构
4. 编写数据隐私合规检查工具
5. 实现请求缓存和批处理优化

